### Importação de Bibliotecas

In [ ]:
import numpy as np
from scipy.io import wavfile
from scipy import signal
import matplotlib.pyplot as plt
import IPython.display as ipd
from google.colab import drive
drive.flush_and_unmount()  # desmonta
drive.mount('/content/drive')  # monta de novo

In [ ]:
# Função fornecida para calcular e plotar FFT
def calcula_plot_FFT(y, fs, titulo="Espectro de Magnitude"):
    """
    Calcula e plota a FFT de um sinal real
    y: sinal de entrada
    fs: frequência de amostragem
    """
    # Converter para mono se for estéreo
    if len(y.shape) > 1:
        y = np.mean(y, axis=1)

    N = len(y)
    Y = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(N, 1/fs)

    plt.figure(figsize=(12, 4))
    plt.plot(freqs, 20*np.log10(np.abs(Y) + 1e-10))
    plt.xlabel('Frequência (Hz)')
    plt.ylabel('Magnitude (dB)')
    plt.title(titulo)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return Y, freqs

# Questão 1

### a) Leitura dos áudios

In [ ]:
# ============================================================================
# QUESTÃO 1 - FFT e Projeto de Filtros
# ============================================================================

print("="*80)
print("QUESTÃO 1 - FFT E PROJETO DE FILTROS")
print("="*80)

# (a) Leitura dos arquivos de áudio
print("\n(a) Lendo arquivos de áudio...")

fs_bird, bird = wavfile.read('/content/drive/MyDrive/Engenharia De Telecomunicações - 88 - Eduardo de Andrade Paziani Kouguem (2023)/2025/6° Semestre/PDS/bird.wav')
fs_noise, noise = wavfile.read('/content/drive/MyDrive/Engenharia De Telecomunicações - 88 - Eduardo de Andrade Paziani Kouguem (2023)/2025/6° Semestre/PDS/noise.wav')

# Converter para mono se necessário
if len(bird.shape) > 1:
    bird = np.mean(bird, axis=1).astype(bird.dtype)
if len(noise.shape) > 1:
    noise = np.mean(noise, axis=1).astype(noise.dtype)

print(f"Bird.wav: taxa de amostragem = {fs_bird} Hz, amostras = {len(bird)}")
print(f"Noise.wav: taxa de amostragem = {fs_noise} Hz, amostras = {len(noise)}")

# Ouvir os arquivos originais
print("\nÁudio original - Pássaro:")
display(ipd.Audio(bird, rate=fs_bird))
print("\nÁudio original - Ruído:")
display(ipd.Audio(noise, rate=fs_noise))

### b) Cálculo dos Espectros

In [ ]:
# (b) Cálculo dos espectros
print("\n" + "="*80)
print("(b) Análise espectral dos sinais")
print("="*80)

Y_bird, freqs_bird = calcula_plot_FFT(bird, fs_bird, "Espectro - Canto do Pássaro")
Y_noise, freqs_noise = calcula_plot_FFT(noise, fs_noise, "Espectro - Ruído de Construção")

print("""
DISCUSSÃO DOS ESPECTROS:

Canto do Pássaro (bird.wav):
- Observa-se que o canto do pássaro possui componentes de frequência concentradas
  principalmente em frequências ALTAS (tipicamente acima de 2-3 kHz)
- Este é um comportamento típico de cantos de pássaros, que possuem sons agudos
- A energia do sinal está concentrada em uma faixa de frequência mais estreita

Ruído de Construção (noise.wav):
- O ruído de construção apresenta componentes principalmente em frequências BAIXAS
- Sons de máquinas, martelos e equipamentos pesados geram ruídos graves
- A energia está distribuída em uma faixa mais ampla, mas com predominância em baixas frequências

Esta diferença na ocupação espectral será crucial para o projeto do filtro passa-altas.
""")

### c) Sinal somado

In [ ]:
# (c) Sinal somado (gravação simulada)
print("\n" + "="*80)
print("(c) Simulação do sinal gravado com ruído")
print("="*80)

# Normalizar os sinais para evitar clipping ao somar
bird_norm = bird.astype(float) / np.max(np.abs(bird))
noise_norm = noise.astype(float) / np.max(np.abs(noise))

# Ajustar comprimentos se necessário
min_len = min(len(bird_norm), len(noise_norm))
bird_norm = bird_norm[:min_len]
noise_norm = noise_norm[:min_len]

# Somar os sinais
sinal_somado = bird_norm +  noise_norm
sinal_somado = sinal_somado / np.max(np.abs(sinal_somado))  # Normalizar

Y_somado, freqs_somado = calcula_plot_FFT(sinal_somado, fs_bird,
                                           "Espectro - Sinal com Ruído")

print("\nÁudio - Pássaro com ruído de construção:")
display(ipd.Audio(sinal_somado, rate=fs_bird))

### d) Filtro IIR PA

In [ ]:
# (d) Projeto dos filtros IIR passa-altas
print("\n" + "="*80)
print("(d) Projeto de Filtros IIR Passa-Altas")
print("="*80)

# Parâmetros do projeto
order = 6  # Ordem do filtro
fc = 2000  # Frequência de corte em Hz (baseada na análise espectral)
Wn = fc / (fs_bird / 2)  # Frequência de corte normalizada (Nyquist)
rp = 0.5  # Ripple na banda passante (dB)
rs = 40   # Atenuação na banda de rejeição (dB)

print(f"""
PARÂMETROS ESCOLHIDOS:
- Ordem: {order}
- Frequência de corte: {fc} Hz
- Wn (normalizada): {Wn:.4f}
- Ripple (rp): {rp} dB
- Atenuação (rs): {rs} dB

JUSTIFICATIVA:
- Frequência de corte em {fc} Hz escolhida para separar o ruído (baixas frequências)
  do canto do pássaro (altas frequências)
- Ordem 6 oferece boa atenuação sem excessiva complexidade computacional
- Ripple de {rp} dB mantém distorção aceitável na banda passante
- Atenuação de {rs} dB garante boa rejeição do ruído
""")

# Projetar os três filtros
b_but, a_but = signal.butter(order, Wn, btype='highpass', analog=False)
b_cheb1, a_cheb1 = signal.cheby1(order, rp, Wn, btype='highpass', analog=False)
b_ellip, a_ellip = signal.ellip(order, rp, rs, Wn, btype='highpass', analog=False)

print("Filtros projetados com sucesso!")

###e) Respostas em Frequência

In [ ]:
# (e) Respostas em frequência
print("\n" + "="*80)
print("(e) Análise das Respostas em Frequência")
print("="*80)

# Calcular respostas em frequência
w_but, h_but = signal.freqz(b_but, a_but, worN=8000)
w_cheb1, h_cheb1 = signal.freqz(b_cheb1, a_cheb1, worN=8000)
w_ellip, h_ellip = signal.freqz(b_ellip, a_ellip, worN=8000)

# Converter para Hz
freq_but = w_but * fs_bird / (2 * np.pi)
freq_cheb1 = w_cheb1 * fs_bird / (2 * np.pi)
freq_ellip = w_ellip * fs_bird / (2 * np.pi)

# Plotar
plt.figure(figsize=(14, 5))
plt.plot(freq_but, 20*np.log10(np.abs(h_but)), label='Butterworth', linewidth=2)
plt.plot(freq_cheb1, 20*np.log10(np.abs(h_cheb1)), label='Chebyshev Tipo I', linewidth=2)
plt.plot(freq_ellip, 20*np.log10(np.abs(h_ellip)), label='Elíptico', linewidth=2)
plt.axvline(fc, color='r', linestyle='--', label=f'Freq. Corte ({fc} Hz)')
plt.xlabel('Frequência (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Respostas em Frequência dos Filtros Passa-Altas')
plt.legend()
plt.grid(True)
plt.xlim([0, 10000])
plt.ylim([-80, 5])
plt.tight_layout()
plt.show()

print("""
COMPARAÇÃO DOS FILTROS:

1. BUTTERWORTH:
   - Resposta mais suave e monotônica
   - Sem ripple na banda passante ou de rejeição
   - Transição mais gradual entre bandas
   - Melhor resposta em fase (mais linear)

2. CHEBYSHEV TIPO I:
   - Ripple na banda passante (visível até ±0.5 dB)
   - Transição mais acentuada que Butterworth
   - Melhor rejeição na banda de corte
   - Pode introduzir distorção devido ao ripple

3. ELÍPTICO:
   - Transição MAIS abrupta entre bandas
   - Ripple tanto na banda passante quanto na de rejeição
   - Melhor desempenho em termos de seletividade
   - Maior complexidade computacional
   - Resposta em fase não-linear

Para aplicações de áudio, Butterworth geralmente oferece melhor qualidade sonora.
""")


### f) Aplicação dos Filtros

In [ ]:
# (f) Aplicação dos filtros
print("\n" + "="*80)
print("(f) Filtragem do Sinal")
print("="*80)

# Aplicar cada filtro
filtered_but = signal.lfilter(b_but, a_but, sinal_somado)
filtered_cheb1 = signal.lfilter(b_cheb1, a_cheb1, sinal_somado)
filtered_ellip = signal.lfilter(b_ellip, a_ellip, sinal_somado)

# Plotar espectros filtrados
Y_but, _ = calcula_plot_FFT(filtered_but, fs_bird, "Espectro Filtrado - Butterworth")
Y_cheb1, _ = calcula_plot_FFT(filtered_cheb1, fs_bird, "Espectro Filtrado - Chebyshev I")
Y_ellip, _ = calcula_plot_FFT(filtered_ellip, fs_bird, "Espectro Filtrado - Elíptico")

print("""
OBSERVAÇÕES DOS ESPECTROS FILTRADOS:
- Todos os três filtros removeram efetivamente as componentes de baixa frequência (ruído)
- O filtro Elíptico apresenta a transição mais abrupta
- O filtro Butterworth mantém a suavidade do espectro
- Componentes de alta frequência (canto do pássaro) foram preservadas
""")

### g) Comparação

In [ ]:
# (g) Comparação auditiva
print("\n" + "="*80)
print("(g) Comparação Auditiva dos Resultados")
print("="*80)

print("\nPássaro original (sem ruído):")
display(ipd.Audio(bird_norm, rate=fs_bird))

print("\nFiltrado - Butterworth:")
display(ipd.Audio(filtered_but, rate=fs_bird))

print("\nFiltrado - Chebyshev Tipo I:")
display(ipd.Audio(filtered_cheb1, rate=fs_bird))

print("\nFiltrado - Elíptico:")
display(ipd.Audio(filtered_ellip, rate=fs_bird))

print("""
AVALIAÇÃO DE DESEMPENHO:

Melhor filtro: BUTTERWORTH

JUSTIFICATIVA:
- Remove efetivamente o ruído de baixa frequência
- Preserva melhor a qualidade sonora do canto do pássaro
- Não introduz artefatos audíveis devido à resposta em fase mais linear
- Ausência de ripple evita distorções no áudio
- Para aplicações de áudio, a suavidade é preferível à máxima seletividade

Os filtros Chebyshev e Elíptico, embora mais seletivos, podem introduzir
artefatos perceptíveis devido ao ripple e à resposta em fase não-linear.
""")

# Questão 2

## a) Convolução Circular


In [ ]:
# ============================================================================
# QUESTÃO 2 - Compressão de Áudio com FFT
# ============================================================================

print("\n" + "="*80)
print("QUESTÃO 2 - COMPRESSÃO DE ÁUDIO COM FFT")
print("="*80)

# (a) Leitura dos arquivos
print("\n(a) Lendo arquivos de áudio...")

fs_handel, handel = wavfile.read('/content/drive/MyDrive/Engenharia De Telecomunicações - 88 - Eduardo de Andrade Paziani Kouguem (2023)/2025/6° Semestre/PDS/Handel.wav')
fs_gong, gong = wavfile.read('/content/drive/MyDrive/Engenharia De Telecomunicações - 88 - Eduardo de Andrade Paziani Kouguem (2023)/2025/6° Semestre/PDS/gong.wav')

# Converter para mono se necessário
if len(handel.shape) > 1:
    handel = np.mean(handel, axis=1).astype(handel.dtype)
if len(gong.shape) > 1:
    gong = np.mean(gong, axis=1).astype(gong.dtype)

print(f"Gong.wav: taxa = {fs_gong} Hz, amostras = {len(gong)}")
print(f"Handel.wav: taxa = {fs_handel} Hz, amostras = {len(handel)}")

## b) FFT dos espectros originais

In [ ]:
# (b) FFTs originais
print("\n" + "="*80)
print("(b) Espectros Originais")
print("="*80)

Y_gong_orig, freqs_gong = calcula_plot_FFT(gong, fs_gong, "Espectro Original - Gong")
Y_handel_orig, freqs_handel = calcula_plot_FFT(handel, fs_handel,
                                                "Espectro Original - Handel")


### C) Técnica de Compressão

In [ ]:
# (c) Técnica de compressão
print("\n" + "="*80)
print("(c) Aplicação da Técnica de Compressão")
print("="*80)

def comprimir_audio(y, fs, limiar_db):
    """
    Comprime áudio removendo componentes espectrais abaixo de um limiar
    """
    # Converter para mono se necessário
    if len(y.shape) > 1:
        y = np.mean(y, axis=1)

    # Calcular FFT
    Y = np.fft.rfft(y)

    # Converter para dB
    magnitude_db = 20 * np.log10(np.abs(Y) + 1e-10)

    # Encontrar limiar absoluto
    max_db = np.max(magnitude_db)
    limiar_absoluto = max_db - limiar_db

    # Criar máscara
    mascara = magnitude_db > limiar_absoluto

    # Aplicar compressão
    Y_comprimido = Y.copy()
    Y_comprimido[~mascara] = 0

    # Reconstruir sinal
    y_reconstruido = np.fft.irfft(Y_comprimido, n=len(y))

    # Calcular estatísticas
    M = np.sum(mascara)  # Amostras não-nulas
    N = len(Y)
    razao_compressao = (N - M) / N

    return y_reconstruido, Y_comprimido, M, N, razao_compressao

# Escolha de limiares
limiar_gong = 60  # dB abaixo do máximo
limiar_handel = 60  # dB abaixo do máximo

print(f"""
ESCOLHA DOS LIMIARES:

RACIOCÍNIO:
- Observando os espectros em dB, identificamos que componentes com magnitude
  muito baixa (>60 dB abaixo do máximo) contribuem pouco para o sinal perceptível
- Estes componentes podem ser descartados com impacto mínimo na qualidade auditiva
- Limiar de {limiar_gong} dB para o gong
- Limiar de {limiar_handel} dB para Handel

O limiar foi escolhido analisando onde o espectro cai para níveis próximos ao
ruído de quantização e componentes imperceptíveis ao ouvido humano.
""")

# Comprimir os sinais
gong_comp, Y_gong_comp, M_gong, N_gong, razao_gong = comprimir_audio(
    gong, fs_gong, limiar_gong)
handel_comp, Y_handel_comp, M_handel, N_handel, razao_handel = comprimir_audio(
    handel, fs_handel, limiar_handel)


### d) Espectros comprimidos

In [ ]:
# (d) Espectros comprimidos
print("\n" + "="*80)
print("(d) Espectros dos Sinais Comprimidos")
print("="*80)

calcula_plot_FFT(gong_comp, fs_gong, "Espectro Comprimido - Gong")
calcula_plot_FFT(handel_comp, fs_handel, "Espectro Comprimido - Handel")

print("""
DIFERENÇAS EM RELAÇÃO AOS ORIGINAIS:
- Componentes de baixa magnitude foram eliminadas (aparecem como -inf em dB)
- A estrutura geral do espectro foi preservada
- Componentes principais (picos) permanecem intactas
- Redução significativa de dados sem perda perceptível de qualidade
""")

### e) Análise de Compressão

In [ ]:
# (e) Análise de compressão
print("\n" + "="*80)
print("(e) Análise das Razões de Compressão")
print("="*80)

print(f"""
GONG.WAV:
- Total de amostras espectrais (N): {N_gong}
- Amostras mantidas (M): {M_gong}
- Amostras zeradas: {N_gong - M_gong}
- Razão de compressão: {razao_gong:.2%}

HANDEL.WAV:
- Total de amostras espectrais (N): {N_handel}
- Amostras mantidas (M): {M_handel}
- Amostras zeradas: {N_handel - M_handel}
- Razão de compressão: {razao_handel:.2%}

SIGNIFICADO DA RAZÃO DE COMPRESSÃO:
- Representa a porcentagem de amostras que foram descartadas
- Uma razão de {razao_gong:.1%} significa que apenas {(1-razao_gong)*100:.1f}% dos
  coeficientes espectrais são necessários para reconstruir o sinal
- Isto se traduz em economia significativa de armazenamento/transmissão
- Na prática, apenas os índices e valores das amostras não-nulas precisam ser armazenados
- Esta é a base de algoritmos de compressão como MP3, AAC, etc.
""")

### f) Comparação Auditiva

In [ ]:
# (f) Comparação auditiva
print("\n" + "="*80)
print("(f) Comparação Auditiva")
print("="*80)

print("\nGong - Original:")
display(ipd.Audio(gong, rate=fs_gong))
print("\nGong - Comprimido:")
display(ipd.Audio(gong_comp, rate=fs_gong))

print("\nHandel - Original:")
display(ipd.Audio(handel, rate=fs_handel))
print("\nHandel - Comprimido:")
display(ipd.Audio(handel_comp, rate=fs_handel))

print("""
PERCEPÇÃO DE DIFERENÇAS:
- Para a maioria dos ouvintes, os sinais comprimidos são perceptualmente idênticos
- As diferenças, se existirem, são sutis e ocorrem principalmente em:
  * Reverberações de longa duração
  * Componentes de frequência muito baixa amplitude
  * Ruído de fundo

Esta técnica demonstra o princípio da compressão com perdas: removemos informação
que é imperceptível ou pouco importante, mantendo a qualidade percebida.

CONCLUSÃO:
A compressão baseada em FFT é altamente eficaz para sinais de áudio, permitindo
redução de {razao_gong:.0%}-{razao_handel:.0%} no volume de dados com impacto
mínimo na qualidade perceptual.
""")

print("\n" + "="*80)
print(" "*32+"FIM DO EXERCÍCIO")
print("="*80)